# עמידות ונגישות בתחבורה הציבורית בישראל

שאלת המחקר: אילו תחנות או מקטעי נסיעה הם באמת קריטיים לנגישות התחבורה הציבורית בישראל בזמן שיבושים, ואיזה סט קטן של קישורי גיבוי או שיפורי מעבר יכול להפוך את הרשת לעמידה יותר?

המחברת הזו היא נקודת הכניסה להגשה בקורס. המימוש הרב-פעמי נמצא בקובץ `src/transit_resilience_analysis.py`.

## מודל הגרף

- תחנות GTFS סמוכות מאוחדות לתחנה אחת לפי `parent_station` ולפי קרבה גיאוגרפית.
- קשת מכוונת נוצרת כאשר תחנה מאוחדת אחת מופיעה מיד אחרי תחנה מאוחדת אחרת בנסיעה.
- משקל קשת הוא מספר מקטעי הנסיעה המתוזמנים שמשתמשים בחיבור הזה.
- זמן הקשת מוערך מתוך זמני הנסיעה המתוזמנים כאשר הם זמינים.
- מדדי מרכזיות משמשים כקו בסיס, ולאחר מכן נבדקים באמצעות סימולציות שיבוש.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'israel-public-transportation'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
SCRIPT = PROJECT_ROOT / 'src' / 'transit_resilience_analysis.py'

PROJECT_ROOT

## בדיקת נתונים נדרשים

בניית הגרף המלא דורשת את `stop_times.txt`. אם הקובץ חסר, צריך לשחזר אותו מהיסטוריית Git LFS לפני הרצת הפייפליין.

In [ ]:
required = [
    DATA_DIR / 'stops.txt',
    DATA_DIR / 'routes.txt',
    DATA_DIR / 'trips.txt',
    DATA_DIR / 'stop_times.txt',
]
for path in required:
    print(path.name, 'OK' if path.exists() else 'MISSING')

if not (DATA_DIR / 'stop_times.txt').exists():
    print('\nפקודת שחזור:')
    print('git checkout 733a885 -- israel-public-transportation/stop_times.txt')
    print('git lfs pull --include="israel-public-transportation/stop_times.txt"')

## הרצת הפייפליין

הפרמטרים הבאים מיועדים להרצה חקירתית. לתוצאות סופיות כדאי להגדיל את `--betweenness-samples`, את `--accessibility-sample` ואת `--random-trials`.

In [ ]:
cmd = [
    sys.executable, str(SCRIPT),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--betweenness-samples', '32',
    '--accessibility-sample', '48',
    '--resilience-removals', '150',
    '--resilience-steps', '8',
    '--random-trials', '2',
    '--single-disruption-candidates', '50',
    '--single-segment-candidates', '50',
    '--backup-links', '8',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

## סיכום הנתונים והרשת

In [ ]:
summary = json.loads((OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
summary

## אילו תחנות באמת פוגעות בנגישות?

In [ ]:
impact = pd.read_csv(OUTPUT_DIR / 'tables' / 'single_station_disruption_impact.csv')
impact[[
    'station_id', 'station_name', 'reachable_share_loss',
    'largest_component_share_loss', 'weighted_degree',
    'approx_betweenness', 'is_articulation_point'
]].head(20)

## עקומת עמידות

In [ ]:
resilience = pd.read_csv(OUTPUT_DIR / 'tables' / 'resilience_curve.csv')
display(resilience.head())
display(Image(filename=str(OUTPUT_DIR / 'figures' / 'resilience_curve.png')))

## אילו מקטעי נסיעה פוגעים בנגישות?

In [ ]:
segments = pd.read_csv(OUTPUT_DIR / 'tables' / 'single_segment_disruption_impact.csv')
segments[[
    'source_station_id', 'target_station_id',
    'source_station_name', 'target_station_name',
    'frequency', 'is_bridge', 'reachable_share_loss',
    'largest_component_share_loss'
]].head(20)

segment_plot = OUTPUT_DIR / 'figures' / 'single_segment_impact.png'
if segment_plot.exists():
    display(Image(filename=str(segment_plot)))

## המלצות לקישורי גיבוי

In [ ]:
backup = pd.read_csv(OUTPUT_DIR / 'tables' / 'recommended_backup_links.csv')
display(backup.head(20))

backup_map = OUTPUT_DIR / 'figures' / 'backup_links_map.png'
if backup_map.exists():
    display(Image(filename=str(backup_map)))

## שאלות לפרשנות בדוח

- האם התחנות בעלות המרכזיות הגבוהה ביותר הן גם התחנות שגורמות לאובדן הנגישות הגדול ביותר?
- איזו אסטרטגיית הסרה מזיקה יותר מהסרה אקראית?
- האם נקודות חיתוך מזיקות יותר מתחנות בעלות תדירות שירות גבוהה?
- האם קישורי גיבוי משחזרים נגישות משמעותית, או רק מחברים רכיבים קטנים ומבודדים?
- אילו מגבלות נובעות מכך ש-GTFS הוא נתון לוחות זמנים ולא נתון ביקוש נוסעים?